In [0]:
# 04_sac_export — Stage 6: export the gold layer for SAC consumption
# Reads the two governed gold tables and writes them as clean single-file CSVs.
# These CSVs are imported into SAP Analytics Cloud as datasets (Option C2).

# --- Cell 1: confirm the gold tables before exporting (Step 2 sanity check) ---
spark.sql("""
    SELECT 'vendor'   AS table_name, COUNT(*) AS row_count
    FROM workspace.default.gold_vendor_performance
    UNION ALL
    SELECT 'cashflow' AS table_name, COUNT(*) AS row_count
    FROM workspace.default.gold_monthly_cashflow
""").show()

In [0]:
# Cell 2 (new) — vendor gold
vendor_df = spark.table("workspace.default.gold_vendor_performance")
display(vendor_df)

In [0]:
# Cell 3 (new) — cashflow gold
cashflow_df = spark.table("workspace.default.gold_monthly_cashflow")
display(cashflow_df)

In [0]:
# --- Cell 3: export gold_monthly_cashflow to a single CSV ---
cashflow_df = spark.table("workspace.default.gold_monthly_cashflow")

(cashflow_df
    .coalesce(1)
    .write
    .option("header", "true")
    .mode("overwrite")
    .csv("/tmp/sac_export/gold_monthly_cashflow"))

print(f"Cash-flow gold exported: {cashflow_df.count()} rows")

In [0]:
# --- Cell 4: locate the actual CSV files so you can download them ---
# Spark writes a folder containing a part-*.csv file. This finds the exact path.

for name in ["gold_vendor_performance", "gold_monthly_cashflow"]:
    files = dbutils.fs.ls(f"/tmp/sac_export/{name}")
    csv_file = [f.path for f in files if f.path.endswith(".csv")][0]
    print(f"{name}:  {csv_file}")